# In Class Assignment 7 — Classifying Penguins with Keras

**Steps:**
1. Load the palmerpenguins dataset and drop rows with missing values.
2. Build the feature matrix from the four numeric measurements plus one-hot encoded sex, then scale to [0, 1].
3. Encode the species column as integer class labels.
4. Split into train and test sets, stratified by species.
5. Define a dense feed-forward network with a softmax output over the three species.
6. Compile with sparse categorical cross-entropy (`from_logits=False`), train, and plot the loss/accuracy curves.
7. Predict on the test set and report accuracy, precision, recall, F1, ROC AUC, the confusion matrix, and a classification report.
8. Repeat with a logits-output model (no final activation, `from_logits=True`) to compare the two parameterisations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, roc_auc_score, classification_report
)

## Step 1 — Load data and drop missing values

In [ ]:
! pip install palmerpenguins -q
from palmerpenguins import load_penguins

penguins = load_penguins()
penguins = penguins.dropna()          # remove rows with any missing value
penguins = penguins.reset_index(drop=True)
print(f"Rows after dropping nulls: {len(penguins)}")
penguins.head()

## Step 2 — Build feature matrix and scale to [0, 1]

In [ ]:
penguins_x = pd.concat([
    penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']],
    pd.get_dummies(penguins['sex'])
], axis=1)

scaler = MinMaxScaler()
X = pd.DataFrame(scaler.fit_transform(penguins_x), columns=penguins_x.columns)
X.head()

## Step 3 — Encode species as integer class labels

In [ ]:
species_cat = penguins['species'].astype('category')
y = species_cat.cat.codes.to_numpy()
label_map = dict(enumerate(species_cat.cat.categories))
print("Label encoding:", label_map)
print("Class distribution:", pd.Series(y).value_counts().sort_index().to_dict())

## Step 4 — Train / test split (stratified by species)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

## Step 5 & 6 — Model 1: softmax output, `from_logits=False`

In [ ]:
inputs = keras.Input(shape=(6,))
x = layers.Dense(16, activation='relu')(inputs)
x = layers.Dense(8,  activation='relu')(x)
outputs = layers.Dense(3, activation='softmax')(x)   # softmax → probabilities

model1 = keras.Model(inputs=inputs, outputs=outputs, name="model_softmax")
model1.summary()

In [ ]:
model1.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer='adam',
    metrics=['accuracy']
)

history1 = model1.fit(
    X_train, y_train,
    epochs=150, batch_size=32,
    validation_split=0.1,
    verbose=0
)

# Plot loss and accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history1.history['loss'],     label='train')
axes[0].plot(history1.history['val_loss'], label='val')
axes[0].set_title('Model 1 — Loss');  axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history1.history['accuracy'],     label='train')
axes[1].plot(history1.history['val_accuracy'], label='val')
axes[1].set_title('Model 1 — Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

## Step 7 — Evaluate Model 1 on the test set

In [ ]:
y_proba1 = model1.predict(X_test)
y_pred1  = np.argmax(y_proba1, axis=1)

print("=== Model 1 (softmax / from_logits=False) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred1):.4f}")
print(f"Precision: {precision_score(y_test, y_pred1, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred1, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred1, average='weighted'):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_proba1, multi_class='ovr'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred1))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred1, target_names=list(label_map.values())))

## Step 8 — Model 2: logits output, `from_logits=True`

Identical architecture but the final Dense layer has **no activation** (outputs raw logits).
The loss must be told `from_logits=True` so it applies softmax internally.

In [ ]:
inputs2  = keras.Input(shape=(6,))
x2 = layers.Dense(16, activation='relu')(inputs2)
x2 = layers.Dense(8,  activation='relu')(x2)
outputs2 = layers.Dense(3)(x2)                    # no activation → raw logits

model2 = keras.Model(inputs=inputs2, outputs=outputs2, name="model_logits")
model2.summary()

In [ ]:
model2.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer='adam',
    metrics=['accuracy']
)

history2 = model2.fit(
    X_train, y_train,
    epochs=150, batch_size=32,
    validation_split=0.1,
    verbose=0
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history2.history['loss'],     label='train')
axes[0].plot(history2.history['val_loss'], label='val')
axes[0].set_title('Model 2 — Loss');  axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history2.history['accuracy'],     label='train')
axes[1].plot(history2.history['val_accuracy'], label='val')
axes[1].set_title('Model 2 — Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
y_logits2 = model2.predict(X_test)
y_proba2  = tf.nn.softmax(y_logits2).numpy()   # convert logits → probabilities for metrics
y_pred2   = np.argmax(y_proba2, axis=1)

print("=== Model 2 (logits / from_logits=True) ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred2):.4f}")
print(f"Precision: {precision_score(y_test, y_pred2, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred2, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred2, average='weighted'):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_proba2, multi_class='ovr'):.4f}")
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred2))
print()
print("Classification Report:")
print(classification_report(y_test, y_pred2, target_names=list(label_map.values())))

## Comparison

Both models implement the **same computation** — `SparseCategoricalCrossentropy` with `from_logits=True` applies softmax inside the loss, while `from_logits=False` expects the network to have already applied softmax. With identical architecture and data the two should converge to similar accuracy.

The key take-away: **never mix `sigmoid`/`softmax` output with `from_logits=True`**, or a linear output with `from_logits=False` — doing so double-applies (or skips) the normalisation and produces garbage predictions (the NaN outputs seen in the starter code).